# transcribe-colab（英語版）: Google Colab GPU 高速文字起こし

英語音声専用のノートブックです。モデルは `Systran/faster-whisper-large-v3`（`LANGUAGE="en"` + `MODEL="auto"` で自動選択、GPU `float16` 優先 / 失敗時は `int8` 自動フォールバック）。

- 入力: 音声ファイル（`.mp3` / `.m4a` / `.wav` / `.mp4`）をまとめた zip を Colab の `/content` にアップロード → 設定の `INPUT_DIR` へ自動展開します（Drive マウントは不要）
- 出力先: 各入力ファイルと同じ階層の `transcribed/` フォルダ（`<元ファイル名>.txt` と `<元ファイル名>.md`）。最後にダウンロード用 zip も作成します
- 事前確認: メニュー「**ランタイム ＞ ランタイムのタイプを変更 ＞ T4 GPU**」が選択されていることを確認してから、上から順にセルを実行してください。


## 1. 設定（入力フォルダ・言語・モデル）

In [ ]:
#@title 文字起こし設定（英語版） { display-mode: "form" }
INPUT_DIR = "/content/kyukyoku_audio"  #@param {type:"string"}
LANGUAGE = "en"  #@param ["en", "auto"]
MODEL = "auto"  #@param {type:"string"}
OVERWRITE = False  #@param {type:"boolean"}

# MODEL = "auto" のとき、LANGUAGE = "en" から Systran/faster-whisper-large-v3 が自動選択される
# Colab 上で直接このノートブックを開いた場合に src/transcribe.py を取得するリポジトリ URL
REPO_URL = "https://github.com/imim100913-cell/transcribe-colab.git"


## 2. 音声 zip の展開と GPU 確認

In [ ]:
import subprocess
import zipfile
from pathlib import Path

INPUT_PATH = Path(INPUT_DIR)
AUDIO_EXTS = {".mp3", ".m4a", ".wav", ".mp4"}


def _audio_count(base: Path) -> int:
    if not base.is_dir():
        return 0
    return sum(
        1
        for p in base.rglob("*")
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS and "transcribed" not in p.parts
    )


def _extract_zip(zip_path: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    print(f"[INFO] 展開しました: {zip_path.name} -> {dest}")


if _audio_count(INPUT_PATH) > 0:
    print(f"[INFO] 入力フォルダに音声が既にあります（{_audio_count(INPUT_PATH)} 件）: {INPUT_DIR}")
else:
    zips = sorted(Path("/content").glob("*.zip"))
    if zips:
        for zp in zips:
            _extract_zip(zp, INPUT_PATH)
    else:
        try:
            from google.colab import files

            print("[INFO] /content に zip が無いため、アップロードダイアログを開きます。音声をまとめた zip を選択してください。")
            uploaded = files.upload()
            for name, data in uploaded.items():
                zp = Path("/content") / name
                zp.write_bytes(data)
                if zp.suffix.lower() == ".zip":
                    _extract_zip(zp, INPUT_PATH)
                else:
                    INPUT_PATH.mkdir(parents=True, exist_ok=True)
                    zp.replace(INPUT_PATH / zp.name)
        except ImportError:
            print("[WARN] Google Colab 環境外のためアップロードをスキップしました。")
    print(f"[INFO] 入力フォルダの音声件数: {_audio_count(INPUT_PATH)} 件")

res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if res.returncode == 0 and res.stdout.strip():
    print(f"[INFO] GPU 検出: {res.stdout.strip()}")
else:
    print("[WARN] GPU が検出されません。Colab の『ランタイム > ランタイムのタイプを変更 > T4 GPU』を確認してください。")


## 3. 依存関係のインストールと本体コード (`src/transcribe.py`) の準備

In [ ]:
import subprocess
import sys
from pathlib import Path

# --- 環境セットアップ（2026-11-22 実機検証で判明した2件の罠への対策） ---
# 罠1: Colab のランタイムは CUDA 13 系のみで、ctranslate2 が必要とする
#       libcublas.so.12 等が無い → pip の nvidia-*-cu12 で補い ldconfig に登録する。
# 罠2: PyAV 19 系は av.open() の metadata_errors 引数が無く faster-whisper 1.2.1 と
#       非互換 → ローカル版で実績のある av==16.1.0 に固定。固定できない環境では
#       decode のスモークテストで検出し、インストール済み audio.py へ互換パッチを当てる。
import glob
import site

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("nvidia-cublas-cu12", "nvidia-cudnn-cu12", "nvidia-cuda-runtime-cu12")
_lib_dirs = sorted({str(p) for base in site.getsitepackages() for p in Path(base).glob("nvidia/*/lib")})
if _lib_dirs:
    Path("/etc/ld.so.conf.d/nvidia-cu12.conf").write_text("\n".join(_lib_dirs) + "\n")
    subprocess.run(["ldconfig"], check=False)
    print(f"[INFO] CUDA 12 ライブラリを ldconfig に登録: {_lib_dirs}")
else:
    print("[WARN] nvidia ライブラリの場所が見つかりませんでした")

try:
    _pip("faster-whisper==1.2.1", "av==16.1.0")
    print("[INFO] faster-whisper==1.2.1 / av==16.1.0 をインストール")
except subprocess.CalledProcessError:
    _pip("faster-whisper==1.2.1")
    print("[WARN] av==16.1.0 を固定できなかったため faster-whisper 単体で続行（後段で互換パッチを判定）")

# decode_audio のスモークテスト（1秒の無音 wav、別プロセスで確認）
import wave
_test_wav = "/tmp/_fw_decode_test.wav"
with wave.open(_test_wav, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(16000)
    w.writeframes(b"\x00\x00" * 16000)

def _decode_ok() -> bool:
    code = (
        "from faster_whisper.audio import decode_audio;"
        f"decode_audio(r'{_test_wav}');"
        "print('decode ok')"
    )
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print("[INFO] decode check:", r.stderr.strip().splitlines()[-1])
    return r.returncode == 0

if not _decode_ok():
    _loc = subprocess.run(
        [sys.executable, "-c", "import faster_whisper.audio as m; print(m.__file__)"],
        capture_output=True, text=True,
    )
    _audio_py = Path(_loc.stdout.strip())
    _src = _audio_py.read_text(encoding="utf-8")
    _patched = _src.replace(', metadata_errors="ignore"', "").replace('metadata_errors="ignore", ', "").replace('metadata_errors="ignore"', "")
    if _patched != _src:
        _audio_py.write_text(_patched, encoding="utf-8")
        print("[INFO] faster_whisper/audio.py の metadata_errors 引数を除去しました")
    if not _decode_ok():
        raise RuntimeError("decode_audio のスモークテストに失敗しました。ランタイムを再起動してから再実行してください。")
print("[INFO] decode_audio スモークテスト OK")

import ctypes
try:
    ctypes.CDLL("libcublas.so.12")
    print("[INFO] libcublas.so.12 ロード OK")
except OSError as e:
    print(f"[WARN] libcublas.so.12 をロードできません: {e}")

# カレントディレクトリに src/transcribe.py が無ければ GitHub からクローン
local_repo = Path.cwd()
if not (local_repo / "src" / "transcribe.py").is_file():
    colab_repo = Path("/content/transcribe-colab")
    if colab_repo.exists():
        subprocess.run(["git", "-C", str(colab_repo), "pull", "--ff-only"], check=False)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(colab_repo)])
    repo_root = colab_repo
else:
    repo_root = local_repo

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import ctranslate2
print(f"[INFO] repo_root = {repo_root}")
print(f"[INFO] CUDA device count = {ctranslate2.get_cuda_device_count()}")


## 4. 文字起こしの実行

In [ ]:
from src.transcribe import transcribe_directory

summary = transcribe_directory(
    input_dir=INPUT_DIR,
    language=LANGUAGE,
    model=MODEL,
    overwrite=OVERWRITE,
)


## 5. 結果確認（サマリーと先頭出力のプレビュー）とダウンロード用 zip

In [ ]:
print("=== 実行結果サマリー ===")
print(f"入力フォルダ : {summary.input_dir}")
print(f"使用モデル   : {summary.model_name}")
print(f"デバイス     : {summary.device} ({summary.compute_type})")
print(
    f"件数         : total={summary.total_files}, "
    f"processed={summary.processed_count}, "
    f"skipped={summary.skipped_count}, "
    f"failed={summary.failed_count}"
)
print(f"総処理時間   : {summary.elapsed_seconds:.1f}s")

failures = [r for r in summary.results if r.status == "failed"]
if failures:
    print("\n[ERROR] 失敗したファイル:")
    for r in failures:
        print(f"  - {r.audio_path}: {r.error}")

preview_targets = [r for r in summary.results if r.status == "processed"] or [
    r for r in summary.results if r.status == "skipped" and r.md_path.is_file()
]
if preview_targets:
    first = preview_targets[0]
    print(f"\n--- プレビュー (.md): {first.md_path} ---")
    print("\n".join(first.md_path.read_text(encoding="utf-8").splitlines()[:20]))
    print(f"\n--- プレビュー (.txt): {first.txt_path} ---")
    print("\n".join(first.txt_path.read_text(encoding="utf-8").splitlines()[:10]))


# --- ダウンロード用に transcribed/ の出力を zip 化 ---
import zipfile
from pathlib import Path as _Path

_zip_path = _Path("/content/kyukyoku_transcripts.zip")
_input_path = _Path(INPUT_DIR)
_files = (
    sorted(p for p in _input_path.rglob("transcribed/*") if p.is_file())
    if _input_path.is_dir()
    else []
)
if _files:
    with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in _files:
            zf.write(p, p.relative_to(_input_path))
    print(f"\n[INFO] ダウンロード用 zip を作成しました: {_zip_path}（{len(_files)} ファイル）")
    try:
        from google.colab import files as _colab_files

        _colab_files.download(str(_zip_path))
    except Exception as _e:
        print(f"[INFO] 自動ダウンロードはスキップされました（{_e}）。左のファイルパネルから {_zip_path.name} を手動でダウンロードしてください。")
else:
    print("\n[INFO] transcribed/ に出力が無いため zip は作成しませんでした。")
